# Hyperparameters

In [ ]:
# %pip install pandas scikit-learn stopwordsiso datasets ipywidgets xgboost mlflow hf_xet tiktoken sentencepiece

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
import stopwordsiso as stopwords
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import mlflow.sklearn

tqdm.pandas()

In [ ]:
# configs
data_path = "../datasets/dataset/full_text_dataset.csv"
ml_flow_url = "https://mlflow.se/"
ml_flow_exp_name = "tfidf-gridsearch"

### Setup experiment and MlFlow things

In [ ]:
# Setup experiment and MlFlow things
mlflow.set_tracking_uri(ml_flow_url)
mlflow.set_experiment(ml_flow_exp_name)

### Create stopwords

In [ ]:
nlp_stopwords = stopwords.stopwords("sv")
nlp_stopwords = set(nlp_stopwords)

# Read custom stopwords from a file
with open("custom_stopwords.txt", "r", encoding="utf-8") as f:
    custom_stopwords = set(line.strip() for line in f if line.strip())

# Combine both sets
stop_words = nlp_stopwords.union(custom_stopwords)
# stop_words

### Load dataset

In [ ]:
df = pd.read_csv(data_path)
len(df)

### Data Preparation Using New Columns to Retain Original Data

In [ ]:
# Initialize columns for labels to keep source data
df['label'] = df['class']
df['label_ids'] = df['class']
df.head()

### Remove empty rows

In [ ]:
# view empty rows
# nan_rows = df[df.isna().any(axis=1)]
# nan_rows

In [ ]:
# Remove rows where 'text' is NaN or empty
df = df[df['text'].notna() & (df['text'] != '')]
len(df)

### Prepare data

In [ ]:
# remove rows with that is not
df = df[df['class'] != 'THIS']
len(df)

In [ ]:
# Remove rows where 'class' is 'NEW' and 'page' is 1
df = df[~((df['class'] == 'THAT') & (df['pagenr'] == 1))]
len(df)

### Prepare data on the new label

In [ ]:
# Set the 'class' column to 'INNER' for all rows where the 'BIO' column is 'INNER'
df.loc[df['BIO'] == 'INNER', 'label'] = 'INNER'

In [ ]:
# Count the number of occurrences of each unique value in the 'class' column
df['label'].value_counts()

In [ ]:
# Your list of allowed values
keep_classes = [
    "WHEN",
    "THIS",
    "THAT",
    "BORING"
]
# keep_classes = [
#     "THIS"
# ]

# # Assuming your DataFrame is called df and the column is named 'label'
# df['label'] = np.where(df['label'].isin(keep_classes), df['label'], 'OTHER')
# df['label'].value_counts()

In [ ]:
# Balance the dataset so that each class has at most 5000 examples (randomly selected)
# nMax = 20000
# df = (
#     df.groupby('label', group_keys=False, observed=True)
#       .apply(lambda x: x.sample(n=min(nMax, len(x)), random_state=42))
#       .reset_index(drop=True)
# )
# df['label'].value_counts()

In [ ]:
def preprocess_text(text):
    """
    Preprocess the text by removing dates, numbers, and other unwanted characters.
    
    Args:
        text (str): Input text to preprocess
        
    Returns:
        str: Preprocessed text
    """
    if not isinstance(text, str):
        return []
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove dates (various formats)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    text = re.sub(r'\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b', '', text)
    
    # Remove all numbers
    text = re.sub(r'\b\d+\b', '', text)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and filter stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    return tokens

### Clean up text

In [ ]:
# Add tokens, apply preprocessing to the text column
df['tokens'] = df['text'].progress_apply(preprocess_text)
# Join tokens into string for each row
df['text_joined'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))
df['first_50_joined'] = df['tokens'].apply(lambda tokens: ' '.join(tokens[:50]))

### Convert string labels to integers

In [ ]:
# Initialize the LabelEncoder
le = LabelEncoder()

# This assigns a unique integer to each unique class (e.g., 'bird' → 0, 'cat' → 1, 'dog' → 2)
df['label_ids'] = le.fit_transform(df['label'])
df.head()

### Split dataset

In [ ]:
# Split based on the 'data_split' column
column_split = 'data_split'
column_split = 'type'
test_df = df[df[column_split] == 'test']
# Filter rows where 'data_split' is either 'train' or 'val'
train_df = df[df[column_split].isin(['train', 'val'])]

# features
column_text = 'first_50_joined'
column_text = 'text_joined'
# column_text = 'text'
X_train_raw = train_df[column_text]
X_test_raw = test_df[column_text]

# Target labels
column_label = 'class'
column_label = 'label_ids'
y_train = train_df[column_label]
y_test = test_df[column_label]

### Fit vectorizer on selected data

In [ ]:
len(X_train_raw)

In [ ]:
# remove rows that is
X_train_vec = X_train_raw
# X_train_vec = train_df[train_df['class'] != 'INNER'][column_text]
# X_train_vec = train_df[train_df['class'] != 'OTHER'][column_text]
len(X_train_vec)

### Hyperparameter search

In [ ]:
# Define base estimators
estimators = [
    ('rf', RandomForestClassifier(n_jobs=-1)),
    # ('ridge', RidgeClassifier())
]

# Voting classifier
voting_clf = VotingClassifier(estimators=estimators, voting='hard')

# Pipeline with TF-IDF + VotingClassifier
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', voting_clf)
])

In [ ]:
# Grid search parameters for TF-IDF
params = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_df': [0.7, 0.8, 0.9, 1.0],
    'tfidf__min_df': [1, 10, 100],
    'tfidf__sublinear_tf': [True, False],
    'tfidf__max_features': [None, 5000, 10000, 25000]
}

In [ ]:
# Grid search
grid = GridSearchCV(pipeline, param_grid=params, cv=5, scoring='accuracy')

In [ ]:
# Start MLflow run
with mlflow.start_run() as run:
    # Train model
    grid.fit(X_train_vec, y_train)
    
    # Best model
    best_model = grid.best_estimator_
    
    # Predict
    y_pred = best_model.predict(X_test_raw)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, target_names=le.classes_)
    
    # Log metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_param("estimator", "VotingClassifier(RF + Ridge)")
    mlflow.log_param("best_params", grid.best_params_)
    
    # Optionally log model
    mlflow.sklearn.log_model(best_model, "model")
    
    # Optionally log the classification report as a text artifact
    with open("classification_report.txt", "w") as f:
        f.write(class_report)
    mlflow.log_artifact("classification_report.txt")


    print("Run ID:", run.info.run_id)
    print("Best TF-IDF params:", grid.best_params_)
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:\n", class_report)